In [8]:
import pandas as pd

# Load data
true = pd.read_csv("/kaggle/input/datasets/utkarshsrivastavji/fake-and-true-dataset-for-lstm-model-prediction/Fake.csv")
fake = pd.read_csv("/kaggle/input/datasets/utkarshsrivastavji/fake-and-true-dataset-for-lstm-model-prediction/True.csv")

# Add labels
fake['label'] = 0   # fake news
true['label'] = 1   # real news

# Combine
df = pd.concat([fake, true], ignore_index=True)

# Shuffle
df = df.sample(frac=1).reset_index(drop=True)

# Combine text columns (important for LSTM)
df['content'] = df['title'] + " " + df['text']

# Keep only needed columns
df = df[['content', 'label']]

# Drop null
df = df.dropna()

import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\s+', ' ', text)   # remove extra spaces
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    return text.strip()

df['content'] = df['content'].apply(clean_text)

df = df.drop_duplicates(subset='content')

# Check
print(df.head(5))
print(df['label'].value_counts())



                                             content  label
0  florida mayors press presidential debate moder...      0
1  lol new video emerges of central park trump as...      1
2  eurofighter jet crashes in spain killing pilot...      0
3  just in senior gop rep announces his retiremen...      1
4  trump administration backs looser obamacare wa...      0
label
0    21178
1    17905
Name: count, dtype: int64


In [3]:
df.isnull().sum()


content    0
label      0
dtype: int64

In [9]:
X=df['content']
Y=df['label']

from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=42,shuffle=True,stratify=Y)




In [10]:

from tensorflow.keras.preprocessing.text import Tokenizer
token = Tokenizer(num_words=20000, oov_token="<OOV>")
token.fit_on_texts(X_train)



In [11]:
X_train_seq = token.texts_to_sequences(X_train)
X_test_seq = token.texts_to_sequences(X_test)


In [12]:
import numpy as np

from tensorflow.keras.preprocessing.sequence import pad_sequences
lengths = [len(seq) for seq in X_train_seq]
print("Max:", max(lengths))
print("Mean:", np.mean(lengths))
print("95 percentile:", np.percentile(lengths, 95))

Max: 8135
Mean: 409.5904177061345
95 percentile: 887.0


In [13]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

X_train_pad = pad_sequences(X_train_seq, maxlen=1000, padding='pre')
X_test_pad = pad_sequences(X_test_seq, maxlen=1000, padding='pre')


In [14]:
!pip install keras-tuner
import keras_tuner as kt
from tensorflow.keras.optimizers import Adam


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [15]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
def build_model(hp):
    model = Sequential()

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256]),
    ))

    model.add(LSTM(
        units=hp.Choice('lstm_units', [32, 64, 128])
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=Adam(
            learning_rate=hp.Choice('lr', [1e-3, 5e-4, 1e-4])
        ),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [16]:
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=5,
    directory='tuner_utkaR',
    project_name='fake_news'
)


2026-02-24 17:36:02.076839: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
tuner.search(
    X_train_pad,
    Y_train,
    epochs=3,
    validation_split=0.2
)

Trial 5 Complete [00h 18m 35s]
val_accuracy: 0.988007664680481

Best val_accuracy So Far: 0.988007664680481
Total elapsed time: 01h 09m 40s


In [17]:
#best_model = tuner.get_best_models(1)[0]
#best_model.summary()    DONE AND THE BEST PARAMS ARE embedding_dim = 64
#lstm_units = 128
#dropout = 0.3
#lr = 0.0005



IndexError: list index out of range

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=64, input_length=540))

# LSTM
model.add(LSTM(128, dropout=0.3, recurrent_dropout=0.3))

# Dense
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.3))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

/usr/local/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [14]:
model = tuner.get_best_models(1)[0]

loss, acc = model.evaluate(X_test_pad, Y_test)
print("Test Accuracy:", acc)

245/245 ━━━━━━━━━━━━━━━━━━━━ 44s 179ms/step - accuracy: 0.9876 - loss: 0.0429
Test Accuracy: 0.9875911474227905


In [19]:

print(df.duplicated(subset='content').sum())


0


In [20]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.optimizers import Adam

model = Sequential()

# Embedding
model.add(Embedding(input_dim=20000, output_dim=128, input_length=540))

# Single strong BiLSTM (faster + stable)
model.add(Bidirectional(LSTM(32, dropout=0.3, recurrent_dropout=0.3)))

# Dense layer (feature extraction)
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.5))

# Output
model.add(Dense(1, activation='sigmoid'))

# Compile
model.compile(
    loss='binary_crossentropy',
    optimizer=Adam(learning_rate=0.0005),
    metrics=['accuracy']
)

In [21]:
from tensorflow.keras.optimizers import Adam, RMSprop

def build_model(hp):
    model = Sequential()

    model.add(Input(shape=(500,)))

    model.add(Embedding(
        input_dim=20000,
        output_dim=hp.Choice('embedding_dim', [64, 128, 256])
    ))

    model.add(Bidirectional(
        LSTM(hp.Choice('lstm_units', [32, 64, 128]))
    ))

    model.add(Dropout(
        hp.Choice('dropout', [0.2, 0.3, 0.5])
    ))

    model.add(Dense(1, activation='sigmoid'))

    # 🔥 optimizer tuning
    optimizer_choice = hp.Choice('optimizer', ['adam', 'rmsprop'])

    lr = hp.Choice('lr', [1e-3, 5e-4, 1e-4])

    if optimizer_choice == 'adam':
        optimizer = Adam(learning_rate=lr)
    else:
        optimizer = RMSprop(learning_rate=lr)

    model.compile(
        optimizer=optimizer,
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [18]:
best_hp = tuner.get_best_hyperparameters(1)[0]
print(best_hp.values)

{'embedding_dim': 64, 'lstm_units': 128, 'dropout': 0.3, 'lr': 0.0005}


In [19]:
best_model = tuner.hypermodel.build(best_hp)

In [22]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=2,
    restore_best_weights=True
)

history = model.fit(
    X_train_pad,
    Y_train,
    epochs=5,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop]
)

Epoch 1/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 219s 548ms/step - accuracy: 0.9122 - loss: 0.2248 - val_accuracy: 0.9728 - val_loss: 0.0789
Epoch 2/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 212s 542ms/step - accuracy: 0.9770 - loss: 0.0788 - val_accuracy: 0.9720 - val_loss: 0.0916
Epoch 3/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 212s 543ms/step - accuracy: 0.9843 - loss: 0.0552 - val_accuracy: 0.9759 - val_loss: 0.0776
Epoch 4/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 212s 543ms/step - accuracy: 0.9888 - loss: 0.0386 - val_accuracy: 0.9685 - val_loss: 0.0960
Epoch 5/5
391/391 ━━━━━━━━━━━━━━━━━━━━ 210s 538ms/step - accuracy: 0.9905 - loss: 0.0335 - val_accuracy: 0.9746 - val_loss: 0.0817


In [23]:
model.evaluate(X_test_pad, Y_test)

245/245 ━━━━━━━━━━━━━━━━━━━━ 20s 81ms/step - accuracy: 0.9779 - loss: 0.0667


[0.06671755760908127, 0.9778687357902527]

In [27]:
model.save("fake_news_model.h5")

import pickle
with open("token.pkl", "wb") as f:
    pickle.dump(token, f)

In [23]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import re

def predict_news(text):
    # cleaning (same as training)
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)

    # convert to sequence
    seq = token.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=540)

    # prediction
    pred = model.predict(padded)[0][0]

    # result
    if pred >= 0.5:
        print("🟢 Real News")
    else:
        print("🔴 Fake News")

    print("Confidence:", float(pred))

In [24]:
news = input("Enter news: ")
predict_news(news)

Enter news:  yoyu


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 498ms/step
🟢 Real News
Confidence: 0.9990605711936951


In [29]:
padded = pad_sequences(seq, maxlen=540)
print(padded)

NameError: name 'seq' is not defined

In [33]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_news(text):
    text = clean_text(text)

    seq = tokenizer.texts_to_sequences([text])
    print("Sequence:", seq)   # DEBUG

    padded = pad_sequences(seq, maxlen=540)
    print("Padded sum:", padded.sum())  # DEBUG

    pred = model.predict(padded)[0][0]

    if pred >= 0.5:
        print("🟢 Real News")
    else:
        print("🔴 Fake News")

    print("Confidence:", float(pred))

In [30]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

In [42]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_news(text):
    text = clean_text(text)

    seq = token.texts_to_sequences([text])
    print("Sequence:", seq)   # DEBUG

    padded = pad_sequences(seq, maxlen=540)
    print("Padded sum:", padded.sum())  # DEBUG

    pred = model.predict(padded)[0][0]

    threshold = 0.6   # try 0.6 or 0.7

    if pred > threshold:
        print("Fake News")
    else:
        print("Real News")

In [49]:
news = input("Enter news: ")
predict_news(news)

Enter news:  Stock markets saw moderate gains on Tuesday as investors reacted positively to recent economic data. Analysts noted that improved consumer spending and stable inflation have contributed to the upward trend. Experts remain cautious, however, due to ongoing global uncertainties.


Sequence: [[2299, 1701, 967, 2136, 3344, 9, 188, 19, 1691, 6244, 9852, 3, 355, 345, 789, 1868, 1389, 8, 4520, 2586, 652, 6, 4068, 4880, 25, 4038, 3, 2, 16833, 3910, 1027, 913, 5972, 326, 539, 3, 1884, 612, 18194]]
Padded sum: 104200
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 81ms/step
Real News


In [50]:
model.save("fake_news_model.h5")

import pickle
with open("token.pkl", "wb") as f:
    pickle.dump(token, f)

In [40]:
print(Y_train.value_counts())

label
0    16942
1    14324
Name: count, dtype: int64
